# 06 — Dummy baseline on the selected features

Goal: the floor for the models track — `DummyClassifier(strategy="most_frequent")`
on the shortlist that notebook 04 persisted to `outputs/selected_features.json`,
selected by name with `feature_set="selected"`.

This notebook does no EDA, feature engineering, or encoding. It loads the shared
artifacts (data, folds) and runs a model through the standard pipeline.
`run_experiment` resolves the named feature set itself — the column list stays
with the feature-selection owner's artifact. Every saved run goes to
`outputs/runs/` and appends a row to `outputs/leaderboard.csv`.


In [ ]:
import pandas as pd

from daa.config import OUTPUTS_DIR
from daa.data import load_train
from daa.experiment import run_experiment
from daa.splits import get_folds


In [ ]:
X, y = load_train()
folds = get_folds(y)
print(f"real data: {X.shape[0]:,} rows x {X.shape[1]:,} columns")
print(f"{len(folds)} folds, {y.nunique()} classes")


## The named feature set

`feature_set="selected"` tells `run_experiment` to resolve the shortlist from
`outputs/selected_features.json` (`daa.features.selection.resolve_feature_set`).
The loader re-checks the column and target fingerprints, so a stale or
hand-edited shortlist fails loudly. The run records `feature_set` and a
`feature_hash` in `params`, so the leaderboard shows which columns it used.


## The dumbest possible baseline

`run_experiment` builds the shared pipeline (preprocessor + model), runs it on
the frozen folds, and records the run. The preprocessor is the default variant
(`features.preprocess.DEFAULT_VARIANT`) and the columns come from
`feature_set="selected"`.

`DummyClassifier` always predicts the majority class. It is the floor: any real
model on the same shortlist must beat it on macro F1.


In [ ]:
dummy_selected = run_experiment(
    "dummy",
    name="dummy_selected",
    X=X,
    y=y,
    folds=folds,
    feature_set="selected",
    fit_full=True,
)
pd.Series(dummy_selected.metrics, name=dummy_selected.name)


## Where the run went

The rows that matter are this run and notebook 04's run on the same shortlist:
their gap on macro F1 is the signal the shortlist carries over the majority
class. Both rows record the same `feature_hash` in `params`.


In [ ]:
leaderboard = pd.read_csv(OUTPUTS_DIR / "leaderboard.csv")
score_columns = ["run_id", "name", "n_features", "macro_f1_mean", "macro_f1_std"]
leaderboard[leaderboard["name"].str.contains("dummy|selected", case=False)][
    score_columns
]


## Next

Swap `"dummy"` for a real model from `daa.models.registry` and keep everything
else — same `feature_set`, same frozen folds — so the leaderboard compares
models, not pipelines. Add `search="grid"` for a tuned variant, or
`repeats=1, save=False` for a fast check. Preprocessing variants live in
`daa.features.preprocess`; encoding experiments belong to the encoding track,
not here.
